1. PassengerID : 탑승객 고유 아이디
2. Survival : 탑승객 생존 유무 (0: 사망, 1: 생존)
3. Pclass : 등실의 등급
4. Name : 이름
5. Sex : 성별
6. Age : 나이
7. Sibsp : 함께 탐승한 형제자매, 아내, 남편의 수
8. Parch : 함께 탐승한 부모, 자식의 수
9. Ticket :티켓 번호
10. Fare : 티켓의 요금
11. Cabin : 객실번호
12. Embarked : 배에 탑승한 항구 이름 ( C = Cherbourn, Q = Queenstown, S = Southampton)

## 라이브러리 가져오기

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

## 데이터 가져오기

In [2]:
train = pd.read_csv('1.titanic_train.csv')
test = pd.read_csv('2.titanic_test.csv')
submission = pd.read_csv('3.titanic_submission.csv')

train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,24,1,1,"Sloper, Mr. William Thompson",male,28.0,0,0,113788,35.5000,A6,S
1,339,1,3,"Dahl, Mr. Karl Edwart",male,45.0,0,0,7598,8.0500,NaN,S
2,769,0,3,"Moran, Mr. Daniel J",male,NaN,1,0,371110,24.1500,NaN,Q
3,692,1,3,"Karun, Miss. Manca",female,4.0,0,1,349256,13.4167,NaN,C
4,891,0,3,"Dooley, Mr. Patrick",male,32.0,0,0,370376,7.7500,NaN,Q


## 전처리에 사용하지 않을 컬럼은 제거

In [ ]:
train = train.drop(['PassengerId', 'Ticket', 'Cabin'], axis=1)
test = test.drop(['PassengerId', 'Ticket', 'Cabin'], axis=1)

## 결측치 처리

In [8]:
train.isna().sum()

Survived      0
Pclass        0
Name          0
Sex           0
Age         146
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

In [9]:
train.Embarked.value_counts()

Embarked
S    538
C    153
Q     64
Name: count, dtype: int64

위 내용을 기준으로 Embarked는 가장 많은 비중을 차지 하는 것이 S이므로 Embarked의 빈값은 S로 채우기로 한다

In [10]:
train['Embarked'] = train.Embarked.fillna('S')
train.isna().sum()

Survived      0
Pclass        0
Name          0
Sex           0
Age         146
SibSp         0
Parch         0
Fare          0
Embarked      0
dtype: int64

### 나이는 쉽게 전부 평균으로 취급하는 것으로

In [14]:
train['Age'] = train.Age.fillna(train.Age.mean())
train

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked
0,1,1,"Sloper, Mr. William Thompson",male,28.000000,0,0,35.5000,S
1,1,3,"Dahl, Mr. Karl Edwart",male,45.000000,0,0,8.0500,S
2,0,3,"Moran, Mr. Daniel J",male,29.646481,1,0,24.1500,Q
3,1,3,"Karun, Miss. Manca",female,4.000000,0,1,13.4167,C
4,0,3,"Dooley, Mr. Patrick",male,32.000000,0,0,7.7500,Q
...,...,...,...,...,...,...,...,...,...
752,0,3,"Boulos, Mrs. Joseph (Sultana)",female,29.646481,0,2,15.2458,C
753,0,3,"Calic, Mr. Petar",male,17.000000,0,0,8.6625,S
754,0,3,"Andersson, Miss. Ebba Iris Alfrida",female,6.000000,4,2,31.2750,S
755,0,3,"Charters, Mr. David",male,21.000000,0,0,7.7333,Q


In [16]:
train.isna().sum()

Survived    0
Pclass      0
Name        0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64

## 생존자와 사망자 데이터셋 분리

In [ ]:
# survived = train[train.Survived == 1]
# dead = train[train.Survived == 0]

### 성별에 따른 생존율 분석

In [ ]:
# pd.DataFrame([survived['Sex'].value_counts(), dead['Sex'].value_counts()], index=[1, 0])
# 여성의 생존율이 압도적으로 높음

Sex,female,male
1,203,87
0,66,401


### pclass 로 생존율 확인

In [ ]:
# pd.DataFrame([survived.groupby('Pclass')['Sex'].count(), 
#               dead.groupby('Pclass')['Sex'].count()], index=[1,0]) 
# # 1등급일 때 생존율이 높고 3등급일 때가 사망율이 높음

Pclass,1,2,3
1,115,72,103
0,70,87,310


In [ ]:
# survived.describe()

,Survived,Pclass,Age,SibSp,Parch,Fare
count,290.0,290.000000,245.000000,290.000000,290.000000,290.000000
mean,1.0,1.958621,28.195918,0.458621,0.482759,49.511940
std,0.0,0.867529,14.973935,0.705891,0.781299,69.593726
min,1.0,1.000000,0.420000,0.000000,0.000000,0.000000
25%,1.0,1.000000,19.000000,0.000000,0.000000,12.381250
50%,1.0,2.000000,28.000000,0.000000,0.000000,26.000000
75%,1.0,3.000000,36.000000,1.000000,1.000000,57.979200
max,1.0,3.000000,63.000000,4.000000,5.000000,512.329200


In [ ]:
# pd.cut(survived.Age, [0,15,40,60,100])

0      (15, 40]
1      (40, 60]
3       (0, 15]
6      (15, 40]
14     (15, 40]
         ...   
742     (0, 15]
743     (0, 15]
748    (15, 40]
750    (15, 40]
751    (15, 40]
Name: Age, Length: 290, dtype: category
Categories (4, interval[int64, right]): [(0, 15] < (15, 40] < (40, 60] < (60, 100]]

## 성별 및 도착지에 대해 원핫인코딩

In [18]:
# 현재 sex, embarked 컬럼은 빈값이 없으므로 바로 원핫인코딩 적용
embarked = pd.get_dummies(train.Embarked)
sex = pd.get_dummies(train.Sex)

In [19]:
df = train.drop(['Embarked', 'Sex'], axis=1)

In [20]:
df = pd.concat([df, embarked, sex], axis=1) # 컬럼 병합
df.head()

,Survived,Pclass,Name,Age,SibSp,Parch,Fare,C,Q,S,female,male
0,1,1,"Sloper, Mr. William Thompson",28.000000,0,0,35.5000,False,False,True,False,True
1,1,3,"Dahl, Mr. Karl Edwart",45.000000,0,0,8.0500,False,False,True,False,True
2,0,3,"Moran, Mr. Daniel J",29.646481,1,0,24.1500,False,True,False,False,True
3,1,3,"Karun, Miss. Manca",4.000000,0,1,13.4167,True,False,False,True,False
4,0,3,"Dooley, Mr. Patrick",32.000000,0,0,7.7500,False,True,False,False,True


## 상관관계 분석

In [ ]:
# abs(df.corr(numeric_only=True).Survived).sort_values() # 절댓값 기준

Q           0.004710
SibSp       0.028497
Age         0.081168
Parch       0.102110
S           0.155452
C           0.171786
Fare        0.257114
Pclass      0.322829
female      0.567451
male        0.567451
Survived    1.000000
Name: Survived, dtype: float64

위 데이터를 봤을 때, 특별한 전처리없이 생존과 관련이 큰 것은 성별과 pclass가 유의미한 값을 보여준다

## 생존자와 사망자간의 차이점 분석

In [ ]:
# df[df.Survived==1].describe()

,Survived,Pclass,Age,SibSp,Parch,Fare
count,290.0,290.000000,245.000000,290.000000,290.000000,290.000000
mean,1.0,1.958621,28.195918,0.458621,0.482759,49.511940
std,0.0,0.867529,14.973935,0.705891,0.781299,69.593726
min,1.0,1.000000,0.420000,0.000000,0.000000,0.000000
25%,1.0,1.000000,19.000000,0.000000,0.000000,12.381250
50%,1.0,2.000000,28.000000,0.000000,0.000000,26.000000
75%,1.0,3.000000,36.000000,1.000000,1.000000,57.979200
max,1.0,3.000000,63.000000,4.000000,5.000000,512.329200


In [ ]:
# df[df.Survived==0].describe()

,Survived,Pclass,Age,SibSp,Parch,Fare
count,467.0,467.000000,366.000000,467.000000,467.000000,467.000000
mean,0.0,2.513919,30.617486,0.520343,0.314775,22.152390
std,0.0,0.742140,14.339948,1.220846,0.805997,32.497167
min,0.0,1.000000,1.000000,0.000000,0.000000,0.000000
25%,0.0,2.000000,21.000000,0.000000,0.000000,7.854200
50%,0.0,3.000000,28.000000,0.000000,0.000000,10.500000
75%,0.0,3.000000,39.750000,1.000000,0.000000,26.000000
max,0.0,3.000000,74.000000,8.000000,6.000000,263.000000


값을 비교해보면 plass가 1에 가까울수록, 지불비용이 높을수록 생존율이 높은 것을 알 수 있다.

In [36]:
train1 = df.drop('Name', axis=1)
train1

,Survived,Pclass,Age,SibSp,Parch,Fare,C,Q,S,female,male
0,1,1,28.000000,0,0,35.5000,False,False,True,False,True
1,1,3,45.000000,0,0,8.0500,False,False,True,False,True
2,0,3,29.646481,1,0,24.1500,False,True,False,False,True
3,1,3,4.000000,0,1,13.4167,True,False,False,True,False
4,0,3,32.000000,0,0,7.7500,False,True,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...
752,0,3,29.646481,0,2,15.2458,True,False,False,True,False
753,0,3,17.000000,0,0,8.6625,False,False,True,False,True
754,0,3,6.000000,4,2,31.2750,False,False,True,True,False
755,0,3,21.000000,0,0,7.7333,False,True,False,False,True


## test 데이터셋 전처리

In [24]:
# 1차 데이터 정리
# name, passengerid, Ticket, Cabin 컬럼은 삭제
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 134 entries, 0 to 133
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Pclass    134 non-null    int64  
 1   Name      134 non-null    object 
 2   Sex       134 non-null    object 
 3   Age       103 non-null    float64
 4   SibSp     134 non-null    int64  
 5   Parch     134 non-null    int64  
 6   Fare      134 non-null    float64
 7   Embarked  134 non-null    object 
dtypes: float64(2), int64(3), object(3)
memory usage: 8.5+ KB


In [26]:
test['Age'] = test.Age.fillna(test.Age.mean())

In [27]:
test.isna().sum()

Pclass      0
Name        0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64

In [28]:
test

,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked
0,2,"Silven, Miss. Lyyli Karoliina",female,18.000000,0,2,13.0000,S
1,1,"Penasco y Castellana, Mrs. Victor de Satode (M...",female,17.000000,1,0,108.9000,C
2,3,"Slocovski, Mr. Selman Francis",male,30.011359,0,0,8.0500,S
3,1,"Silvey, Mrs. William Baird (Alice Munger)",female,39.000000,1,0,55.9000,S
4,2,"Brown, Mr. Thomas William Solomon",male,60.000000,1,1,39.0000,S
...,...,...,...,...,...,...,...,...
129,3,"Green, Mr. George Henry",male,51.000000,0,0,8.0500,S
130,3,"Baclini, Mrs. Solomon (Latifa Qurban)",female,24.000000,0,3,19.2583,C
131,3,"Strandberg, Miss. Ida Sofia",female,22.000000,0,0,9.8375,S
132,2,"Smith, Miss. Marion Elsie",female,40.000000,0,0,13.0000,S


In [29]:
test = test.drop(['Name'], axis=1)
test

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,2,female,18.000000,0,2,13.0000,S
1,1,female,17.000000,1,0,108.9000,C
2,3,male,30.011359,0,0,8.0500,S
3,1,female,39.000000,1,0,55.9000,S
4,2,male,60.000000,1,1,39.0000,S
...,...,...,...,...,...,...,...
129,3,male,51.000000,0,0,8.0500,S
130,3,female,24.000000,0,3,19.2583,C
131,3,female,22.000000,0,0,9.8375,S
132,2,female,40.000000,0,0,13.0000,S


In [32]:
test_sex = pd.get_dummies(test.Sex)
test_embarked = pd.get_dummies(test.Embarked)
test_df = pd.concat([test, test_sex, test_embarked], axis=1)
test_df

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,female,male,C,Q,S
0,2,female,18.000000,0,2,13.0000,S,True,False,False,False,True
1,1,female,17.000000,1,0,108.9000,C,True,False,True,False,False
2,3,male,30.011359,0,0,8.0500,S,False,True,False,False,True
3,1,female,39.000000,1,0,55.9000,S,True,False,False,False,True
4,2,male,60.000000,1,1,39.0000,S,False,True,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...
129,3,male,51.000000,0,0,8.0500,S,False,True,False,False,True
130,3,female,24.000000,0,3,19.2583,C,True,False,True,False,False
131,3,female,22.000000,0,0,9.8375,S,True,False,False,False,True
132,2,female,40.000000,0,0,13.0000,S,True,False,False,False,True


In [33]:
test_df = test_df.drop(['Embarked', 'Sex'], axis=1)
test_df

,Pclass,Age,SibSp,Parch,Fare,female,male,C,Q,S
0,2,18.000000,0,2,13.0000,True,False,False,False,True
1,1,17.000000,1,0,108.9000,True,False,True,False,False
2,3,30.011359,0,0,8.0500,False,True,False,False,True
3,1,39.000000,1,0,55.9000,True,False,False,False,True
4,2,60.000000,1,1,39.0000,False,True,False,False,True
...,...,...,...,...,...,...,...,...,...,...
129,3,51.000000,0,0,8.0500,False,True,False,False,True
130,3,24.000000,0,3,19.2583,True,False,True,False,False
131,3,22.000000,0,0,9.8375,True,False,False,False,True
132,2,40.000000,0,0,13.0000,True,False,False,False,True


### 학습, 타겟 분리

In [38]:
train_df = train1.drop('Survived', axis=1)
train_target = train1.Survived

In [66]:
X_train, X_test, y_train, y_test = train_test_split(train_df, train_target, random_state=42)

In [67]:
from sklearn.ensemble import RandomForestClassifier
rf_clf = RandomForestClassifier()
rf_clf.fit(X_train, y_train)
rf_clf.score(X_train, y_train)

0.9894179894179894

In [86]:
type(y_train)

pandas.core.series.Series

In [78]:
type(p)

numpy.ndarray

In [79]:
type(y_test)

pandas.core.series.Series

In [87]:
p = rf_clf.predict(X_test)

In [90]:
type(y_test)

pandas.core.series.Series

In [92]:
y_test

409    1
97     0
281    1
497    1
440    0
      ..
108    0
744    0
56     1
204    0
234    0
Name: Survived, Length: 190, dtype: int64

In [91]:
pd.Series(p)

0      1
1      0
2      1
3      1
4      0
      ..
185    0
186    0
187    1
188    1
189    0
Length: 190, dtype: int64

In [95]:
len(p == y_test)/len(p)

1.0

In [39]:
from sklearn.ensemble import RandomForestClassifier
clf = RandomForestClassifier(random_state=42)
clf.fit(train_df, train_target)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [40]:
clf.score(train_df, train_target)

0.9867899603698811

In [41]:
train_df

,Pclass,Age,SibSp,Parch,Fare,C,Q,S,female,male
0,1,28.000000,0,0,35.5000,False,False,True,False,True
1,3,45.000000,0,0,8.0500,False,False,True,False,True
2,3,29.646481,1,0,24.1500,False,True,False,False,True
3,3,4.000000,0,1,13.4167,True,False,False,True,False
4,3,32.000000,0,0,7.7500,False,True,False,False,True
...,...,...,...,...,...,...,...,...,...,...
752,3,29.646481,0,2,15.2458,True,False,False,True,False
753,3,17.000000,0,0,8.6625,False,False,True,False,True
754,3,6.000000,4,2,31.2750,False,False,True,True,False
755,3,21.000000,0,0,7.7333,False,True,False,False,True


In [60]:
pred = clf.predict(test_df[['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'C', 'Q', 'S', 'female', 'male']])
pred

array([1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0,
       1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0,
       0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0,
       0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1,
       1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0,
       1, 1])

In [63]:
submission['Survived'] = pred#.astype(bool)
submission

,PassengerId,Survived
0,418,1
1,308,1
2,88,0
3,578,1
4,685,0
...,...,...
129,223,0
130,859,1
131,475,0
132,347,1


In [65]:
submission.to_csv('20250721_test.csv', encoding='utf-8', index=False)

In [57]:
submission.isna().sum()

PassengerId    0
Survived       0
dtype: int64